In [118]:
import geopandas as gpd
import pandas as pd
import numpy as np
import missingno as msno
import os

In [119]:
file_path = '../02_INTERMEDIATE/final_bamboo_parameters_v08.gpkg'
if os.path.exists(file_path):
    gdf = gpd.read_file(file_path)
else:
    print('file not available, check source')

In [120]:
gdf.head()

,psp,year,date_of_collection,clump_id,total_culms,total_agb,max_culm_diameter_cm,avg_culm_diameter_cm,min_culm_diameter_cm,median_culm_diameter_cm,...,bh_hollow_equivalent_diameter_m,bh_hollow_fraction,bh_solid_fraction,bh_outside_area_m2,bh_slice_to_original_area_ratio,bh_n_hollows,bh_slice_valid,bh_n_positive_cells,bh_n_valid_cells,geometry
0,1a-APM-01,2025,4/24/2025,CLUMP #1,11,34.65,2.78,1.86,0.36,1.85,...,0.03,0.0,1.0,2.67,0.92,1.0,1.0,4317.0,82159.0,POINT (798563.99 9777833.328)
1,1a-APM-01,2026,4/2/2026,CLUMP #1,31,144.28,3.59,2.35,0.66,2.45,...,0.04,0.0,1.0,4.45,0.89,2.0,1.0,2532.0,104989.0,POINT (798561.124 9777837.75)
2,1a-APM-01,2025,4/24/2025,CLUMP #2,14,16.47,2.16,1.06,0.39,0.93,...,0.00,0.0,1.0,0.41,0.92,0.0,1.0,829.0,13065.0,POINT (798566.87 9777828.479)
3,1a-APM-01,2026,4/2/2026,CLUMP #2,58,172.09,3.03,1.81,0.23,1.81,...,0.03,0.0,1.0,2.67,0.92,1.0,1.0,4317.0,82159.0,POINT (798563.966 9777833.328)
4,1a-APM-01,2025,4/24/2025,CLUMP #3,0,0.00,0.00,0.00,0.00,0.00,...,0.00,0.0,1.0,1.10,0.91,0.0,1.0,2059.0,29404.0,POINT (798567.766 9777823.46)


In [121]:
gdf.columns

Index(['psp', 'year', 'date_of_collection', 'clump_id', 'total_culms',
       'total_agb', 'max_culm_diameter_cm', 'avg_culm_diameter_cm',
       'min_culm_diameter_cm', 'median_culm_diameter_cm', 'max_culm_agb',
       'avg_culm_agb', 'min_culm_agb', 'median_culm_agb', 'hmin', 'hmax',
       'hmean', 'hstd', 'hmedian', 'total_points', 'canopy_volume_m3',
       'gap_fraction', 'canopy_area', 'val_height', 'val_canopy_diameter',
       'bh_outer_area_m2', 'bh_outer_perimeter_m',
       'bh_outer_equivalent_diameter_m', 'bh_hollow_area_m2',
       'bh_hollow_perimeter_m', 'bh_hollow_equivalent_diameter_m',
       'bh_hollow_fraction', 'bh_solid_fraction', 'bh_outside_area_m2',
       'bh_slice_to_original_area_ratio', 'bh_n_hollows', 'bh_slice_valid',
       'bh_n_positive_cells', 'bh_n_valid_cells', 'geometry'],
      dtype='str')

In [122]:
columns_to_keep = ['psp', 'year', 'date_of_collection', 'clump_id', 'total_culms','total_agb', 
                   'avg_culm_diameter_cm', 'avg_culm_agb', 'hmean', 'canopy_volume_m3',
                    'canopy_area', 'geometry']

In [123]:
gdf = gdf[columns_to_keep]

In [124]:
gdf.head()

,psp,year,date_of_collection,clump_id,total_culms,total_agb,avg_culm_diameter_cm,avg_culm_agb,hmean,canopy_volume_m3,canopy_area,geometry
0,1a-APM-01,2025,4/24/2025,CLUMP #1,11,34.65,1.86,3.15,4.49,180.85,32.87,POINT (798563.99 9777833.328)
1,1a-APM-01,2026,4/2/2026,CLUMP #1,31,144.28,2.35,4.65,5.09,260.88,42.00,POINT (798561.124 9777837.75)
2,1a-APM-01,2025,4/24/2025,CLUMP #2,14,16.47,1.06,1.18,3.72,22.16,5.23,POINT (798566.87 9777828.479)
3,1a-APM-01,2026,4/2/2026,CLUMP #2,58,172.09,1.81,2.97,4.49,180.85,32.87,POINT (798563.966 9777833.328)
4,1a-APM-01,2025,4/24/2025,CLUMP #3,0,0.00,0.00,0.00,2.77,41.65,11.75,POINT (798567.766 9777823.46)


In [125]:
gdf.dtypes

psp                          str
year                       int64
date_of_collection           str
clump_id                     str
total_culms                int64
total_agb                float64
avg_culm_diameter_cm     float64
avg_culm_agb             float64
hmean                    float64
canopy_volume_m3         float64
canopy_area              float64
geometry                geometry
dtype: object

In [126]:
gdf['date_of_collection'].unique()

<StringArray>
['4/24/2025',  '4/2/2026', '4/25/2025',  '5/3/2026', '5/15/2025', '2/20/2026',
 '2/24/2026', '6/18/2025', '3/17/2026', '5/20/2025',  '6/3/2026', '3/13/2026',
 '6/24/2026']
Length: 13, dtype: str

In [127]:
def clean_mixed_dates(val):
    if pd.isna(val):
        return pd.NaT
    
    # Check for three specific broken Day/Month/Year strings
    if val in ['4/2/2026', '5/3/2026', '6/3/2026']:
        return pd.to_datetime(val, format='%d/%m/%Y')
    
    # Default format for everything else (Month/Day/Year)
    return pd.to_datetime(val, format='%m/%d/%Y', errors='coerce')

In [128]:
gdf['date_of_collection']  = gdf['date_of_collection'].apply(clean_mixed_dates)

In [129]:
gdf.dtypes

psp                                str
year                             int64
date_of_collection      datetime64[us]
clump_id                           str
total_culms                      int64
total_agb                      float64
avg_culm_diameter_cm           float64
avg_culm_agb                   float64
hmean                          float64
canopy_volume_m3               float64
canopy_area                    float64
geometry                      geometry
dtype: object

In [130]:
gdf.head(10)

,psp,year,date_of_collection,clump_id,total_culms,total_agb,avg_culm_diameter_cm,avg_culm_agb,hmean,canopy_volume_m3,canopy_area,geometry
0,1a-APM-01,2025,2025-04-24,CLUMP #1,11,34.65,1.86,3.15,4.49,180.85,32.87,POINT (798563.99 9777833.328)
1,1a-APM-01,2026,2026-02-04,CLUMP #1,31,144.28,2.35,4.65,5.09,260.88,42.00,POINT (798561.124 9777837.75)
2,1a-APM-01,2025,2025-04-24,CLUMP #2,14,16.47,1.06,1.18,3.72,22.16,5.23,POINT (798566.87 9777828.479)
3,1a-APM-01,2026,2026-02-04,CLUMP #2,58,172.09,1.81,2.97,4.49,180.85,32.87,POINT (798563.966 9777833.328)
4,1a-APM-01,2025,2025-04-24,CLUMP #3,0,0.00,0.00,0.00,2.77,41.65,11.75,POINT (798567.766 9777823.46)
5,1a-APM-01,2026,2026-02-04,CLUMP #3,11,10.38,0.92,0.94,3.72,22.16,5.23,POINT (798566.861 9777828.486)
6,1a-APM-01,2025,2025-04-24,CLUMP #4,1,0.78,0.91,0.78,NaN,NaN,NaN,POINT (798566.923 9777817.78)
7,1a-APM-01,2026,2026-02-04,CLUMP #4,7,18.30,1.72,2.61,2.77,41.65,11.75,POINT (798567.847 9777823.462)
8,1a-APM-01,2025,2025-04-24,CLUMP #5,0,0.00,0.00,0.00,NaN,NaN,NaN,POINT (798564.058 9777814.547)
9,1a-APM-01,2026,2026-02-04,CLUMP #5,2,0.10,0.20,0.05,NaN,NaN,NaN,POINT (798566.947 9777817.763)


In [131]:
new_names = {'total_culms' : 'number_of_culms', 'total_agb' : 'aboveground_biomass_in_kg',
             'avg_culm_agb' : 'mean_culm_aboveground_biomass', 'hmean' : 'mean_height_metres',
             'canopy_area' : 'canopy_area_m2'
             }

In [132]:
gdf.rename(columns=new_names, inplace=True)

In [133]:
gdf.head()

,psp,year,date_of_collection,clump_id,number_of_culms,aboveground_biomass_in_kg,avg_culm_diameter_cm,mean_culm_aboveground_biomass,mean_height_metres,canopy_volume_m3,canopy_area_m2,geometry
0,1a-APM-01,2025,2025-04-24,CLUMP #1,11,34.65,1.86,3.15,4.49,180.85,32.87,POINT (798563.99 9777833.328)
1,1a-APM-01,2026,2026-02-04,CLUMP #1,31,144.28,2.35,4.65,5.09,260.88,42.00,POINT (798561.124 9777837.75)
2,1a-APM-01,2025,2025-04-24,CLUMP #2,14,16.47,1.06,1.18,3.72,22.16,5.23,POINT (798566.87 9777828.479)
3,1a-APM-01,2026,2026-02-04,CLUMP #2,58,172.09,1.81,2.97,4.49,180.85,32.87,POINT (798563.966 9777833.328)
4,1a-APM-01,2025,2025-04-24,CLUMP #3,0,0.00,0.00,0.00,2.77,41.65,11.75,POINT (798567.766 9777823.46)


In [134]:
is_2025 = gdf['date_of_collection'].dt.year == 2025

# 2. Specify the target columns you want to clear out
target_columns = ['canopy_area_m2', 'canopy_volume_m3', 'mean_height_metres']

# 3. Use .loc[rows, columns] to set those values to np.nan
gdf.loc[is_2025, target_columns] = np.nan

In [135]:
gdf.columns = gdf.columns.str.replace('_', ' ')

In [136]:
gdf.head()

,psp,year,date of collection,clump id,number of culms,aboveground biomass in kg,avg culm diameter cm,mean culm aboveground biomass,mean height metres,canopy volume m3,canopy area m2,geometry
0,1a-APM-01,2025,2025-04-24,CLUMP #1,11,34.65,1.86,3.15,NaN,NaN,NaN,POINT (798563.99 9777833.328)
1,1a-APM-01,2026,2026-02-04,CLUMP #1,31,144.28,2.35,4.65,5.09,260.88,42.00,POINT (798561.124 9777837.75)
2,1a-APM-01,2025,2025-04-24,CLUMP #2,14,16.47,1.06,1.18,NaN,NaN,NaN,POINT (798566.87 9777828.479)
3,1a-APM-01,2026,2026-02-04,CLUMP #2,58,172.09,1.81,2.97,4.49,180.85,32.87,POINT (798563.966 9777833.328)
4,1a-APM-01,2025,2025-04-24,CLUMP #3,0,0.00,0.00,0.00,NaN,NaN,NaN,POINT (798567.766 9777823.46)


In [137]:
gdf.isna().sum()

psp                                0
year                               0
date of collection                 0
clump id                           0
number of culms                    0
aboveground biomass in kg          0
avg culm diameter cm               0
mean culm aboveground biomass      0
mean height metres               131
canopy volume m3                 131
canopy area m2                   131
geometry                           0
dtype: int64

In [138]:
gdf.head()

,psp,year,date of collection,clump id,number of culms,aboveground biomass in kg,avg culm diameter cm,mean culm aboveground biomass,mean height metres,canopy volume m3,canopy area m2,geometry
0,1a-APM-01,2025,2025-04-24,CLUMP #1,11,34.65,1.86,3.15,NaN,NaN,NaN,POINT (798563.99 9777833.328)
1,1a-APM-01,2026,2026-02-04,CLUMP #1,31,144.28,2.35,4.65,5.09,260.88,42.00,POINT (798561.124 9777837.75)
2,1a-APM-01,2025,2025-04-24,CLUMP #2,14,16.47,1.06,1.18,NaN,NaN,NaN,POINT (798566.87 9777828.479)
3,1a-APM-01,2026,2026-02-04,CLUMP #2,58,172.09,1.81,2.97,4.49,180.85,32.87,POINT (798563.966 9777833.328)
4,1a-APM-01,2025,2025-04-24,CLUMP #3,0,0.00,0.00,0.00,NaN,NaN,NaN,POINT (798567.766 9777823.46)


In [139]:
gdf.to_crs('EPSG:4326', inplace=True)

In [140]:
gdf.head()

,psp,year,date of collection,clump id,number of culms,aboveground biomass in kg,avg culm diameter cm,mean culm aboveground biomass,mean height metres,canopy volume m3,canopy area m2,geometry
0,1a-APM-01,2025,2025-04-24,CLUMP #1,11,34.65,1.86,3.15,NaN,NaN,NaN,POINT (29.68377 -2.00779)
1,1a-APM-01,2026,2026-02-04,CLUMP #1,31,144.28,2.35,4.65,5.09,260.88,42.00,POINT (29.68374 -2.00775)
2,1a-APM-01,2025,2025-04-24,CLUMP #2,14,16.47,1.06,1.18,NaN,NaN,NaN,POINT (29.6838 -2.00783)
3,1a-APM-01,2026,2026-02-04,CLUMP #2,58,172.09,1.81,2.97,4.49,180.85,32.87,POINT (29.68377 -2.00779)
4,1a-APM-01,2025,2025-04-24,CLUMP #3,0,0.00,0.00,0.00,NaN,NaN,NaN,POINT (29.6838 -2.00787)


In [141]:
gdf.to_csv('../02_INTERMEDIATE/bamboo_parameters_v_platform.csv', index=False)